In [1]:
# 필요 시 한 번만 실행
# !pip install python-dotenv langchain-openai langchain-core \
#             langchain-community fastembed faiss-cpu

In [7]:
# 환경 변수 로드 및 LLM 설정
import os
import datetime

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores.faiss import FAISS

load_dotenv()  # .env 에서 OPENROUTER_API_KEY 사용

llm = ChatOpenAI(
    model="deepseek/deepseek-chat",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,  # 일부러 약간의 랜덤성 유지
)

In [8]:
# 나이브 체인 (툴만 연결된 느낌, RAG 없음)
naive_prompt = ChatPromptTemplate.from_template(
    """
[역할]
너는 사내 DB/MCP에 연결된 것처럼 답하는 LLM 어시스턴트다.

[조건]
- 실제 DB 스키마, 비즈니스 규칙, 보안 정책은 주어지지 않는다.
- 네가 알고 있는 일반 상식과 추론만으로, 있을 법한 테이블/컬럼을 상상해도 된다.

[해야 할 일]
1) 사용자의 질문을 보고, 내부 DB에 어떤 테이블/컬럼이 있을지 '추측'해서 설명한다.
2) 그에 맞는 SQL 예시를 ```sql ... ``` 블록 안에 작성한다.
3) SQL이 실행되었다고 가정하고, 결과를 한국어로 요약해 답한다.

[질문]
{question}
"""
)


def naive_answer(question: str) -> str:
    return (naive_prompt | llm).invoke({"question": question}).content


In [9]:
# RAG용 비즈니스 문서 & 리트리버 준비
BUSINESS_DOCS = [
    """
[데이터 스키마]

테이블: customer_order_fact

컬럼:
- customer_id : 숫자, 고객 식별자
- seg_code    : 문자, 'V1','V2' = VIP, 'N1' = 일반
- order_amount: 숫자, 주문금액(원)
- order_ym    : 문자, '2025-01' 처럼 연-월

[비즈니스 규칙]

- "VIP 고객"은 seg_code IN ('V1','V2') 인 고객만 의미한다.
- "최근 3개월 평균 주문금액"은 기준월 포함 직전 3개월의 order_amount 평균이다.
  예: 기준월 2025-01 → 2024-11, 2024-12, 2025-01 3개월 기준.
""",
    """
[보안/권한 정책]

- 분석에 주민등록번호(ssn), 계좌번호(account_no), 카드번호(card_no)는 사용하지 않는다.
- 쿼리에서 ssn, account_no, card_no 컬럼을 절대 SELECT 하지 않는다.
- 결과 요약에도 개인을 특정할 수 있는 정보(이름, 주민번호 등)는 포함하지 않는다.
- 보안 위반 요청 시 "정책상 제공 불가" 라고 명확히 안내한다.
"""
]

embeddings = FastEmbedEmbeddings()
vectorstore = FAISS.from_texts(BUSINESS_DOCS, embedding=embeddings)
retriever = vectorstore.as_retriever(k=2)


def get_context(question: str) -> str:
    docs = retriever.invoke(question)
    return "\n\n".join(d.page_content for d in docs)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:02<00:00,  1.78it/s]


In [10]:
# RAG + 정책 체인 (스키마/비즈니스 룰/보안 컨텍스트 주입)
rag_prompt = ChatPromptTemplate.from_template(
    """
[역할]
너는 사내 DW에 연결된 LLM SQL 어시스턴트다.

[컨텍스트: 데이터 스키마 / 비즈니스 규칙 / 보안 정책]
-----------------CONTEXT START-----------------
{context}
------------------CONTEXT END------------------

위 컨텍스트를 반드시 준수해야 한다.
특히 [보안/권한 정책]을 절대 어기지 말 것.

[해야 할 일]
1) 사용자의 질문을 분석해서, 어떤 테이블/컬럼/필터/집계를 사용할지
   한국어로 2~3줄 정도로 계획을 세운다.
2) 그 다음, 아래 형식으로 하나의 SQL만 작성한다.

```sql
SELECT ...
FROM ...
WHERE ...
GROUP BY ...


CONTEXT에 없는 컬럼/테이블 이름은 쓰지 말 것.

ssn, account_no, card_no 같은 민감 컬럼은 절대 SELECT 하지 말 것.

마지막으로, 위 SQL이 정상적으로 실행되어 나온 것처럼
한국어로 한 단락 요약 답변을 작성한다.
(단, CONTEXT에 없는 가정을 새로 만들어내지 말 것.)

[질문]
{question}
"""
)

def rag_answer(question: str) -> str:
    context = get_context(question)
    return (rag_prompt | llm).invoke({"context": context, "question": question}).content


In [11]:
# 간단한 audit log (정책/감사 레이어 흉내)
def write_audit_log(chain_name: str, question: str):
    os.makedirs("logs", exist_ok=True)
    path = os.path.join("logs", "audit.log")
    now = datetime.datetime.now().isoformat()
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"{now} | chain={chain_name} | q={question}\n")

In [14]:
# 데모 실행 (현실 1·2·3 나눠서 보기, 출력 길이 제한)
# 공통: LLM 응답을 앞 몇 줄만 출력하는 헬퍼 함수

def short_print(title: str, text: str, max_lines: int = 8):
    """출력이 너무 길어지지 않도록 앞 부분만 줄 단위로 잘라서 출력."""
    print(f"\n=== {title} ===")
    lines = text.splitlines()
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print("...[이후 줄 생략]...")

In [ ]:
# 현실 1. LLM은 '데이터 구조와 비즈니스 규칙'을 모른다

business_q = "VIP 고객의 2025년 1월 기준 최근 3개월 평균 주문금액을 알려줘."

print("### 현실 1. LLM은 '데이터 구조와 비즈니스 규칙'을 모른다 ###")

# 나이브 체인
write_audit_log("naive", business_q)
naive_res = naive_answer(business_q)
short_print("나이브 체인 (스키마/룰/보안 정책 없음)", naive_res, max_lines=30)

# RAG + 정책 체인
write_audit_log("rag", business_q)
rag_res = rag_answer(business_q)
short_print("RAG + 정책 체인 (스키마/비즈니스 룰/보안 정책 주입)", rag_res, max_lines=30)

### 현실 1. LLM은 '데이터 구조와 비즈니스 규칙'을 모른다 ###

=== 나이브 체인 (스키마/룰/보안 정책 없음) ===
# VIP 고객의 2025년 1월 기준 최근 3개월 평균 주문금액 분석

## 추정 데이터베이스 구조
- `customers` 테이블: 고객 정보
  - `customer_id` (PK)
  - `name`
  - `vip_status` (VIP 여부)
  - `join_date`
  
- `orders` 테이블: 주문 정보
...[이후 줄 생략]...

=== RAG + 정책 체인 (스키마/비즈니스 룰/보안 정책 주입) ===
1) 계획: 
- customer_order_fact 테이블에서 seg_code가 'V1' 또는 'V2'인 VIP 고객을 필터링한다.
- order_ym이 '2024-11', '2024-12', '2025-01'인 데이터를 선택하여 평균 주문금액을 계산한다.

2) SQL:
```sql
SELECT 
    AVG(order_amount) AS avg_order_amount
FROM 
    customer_order_fact
...[이후 줄 생략]...


In [16]:
# 현실 2. '그때그때 다른 결과'의 위험

business_q = "VIP 고객의 2025년 1월 기준 최근 3개월 평균 주문금액을 알려줘."

print("### 현실 2. '그때그때 다른 결과'의 위험 (같은 질문 2번 실행) ###")

for i in range(2):
    write_audit_log("naive", business_q)
    res = naive_answer(business_q)
    short_print(f"나이브 체인 실행 {i+1}", res, max_lines=8)

print(
    "\n※ 같은 질문이어도 온도(temperature), 컨텍스트 상태 등에 따라 "
    "SQL 구조·요약 내용이 달라질 수 있다. "
    "DB/MCP 연결만으로는 '정확성·재현성'이 자동 보장되지 않는다."
)

### 현실 2. '그때그때 다른 결과'의 위험 (같은 질문 2번 실행) ###

=== 나이브 체인 실행 1 ===
# VIP 고객의 2025년 1월 기준 최근 3개월 평균 주문금액 분석

## 추정된 데이터베이스 구조

이 질문에 답하기 위해 필요한 테이블과 컬럼을 추정해보겠습니다:

1. **고객 테이블 (Customers)**
   - `customer_id`: 고객 고유 식별자
   - `name`: 고객 이름
   - `vip_status`: VIP 여부 (예: 'Gold', 'Platinum' 등)
   - `join_date`: 가입일자

2. **주문 테이블 (Orders)**
   - `order_id`: 주문 고유 식별자
   - `customer_id`: 고객 ID (외래키)
   - `order_date`: 주문일자
   - `total_amount`: 주문 총액
   - `status`: 주문 상태 (예: '완료', '취소' 등)

## SQL 쿼리 예시

```sql
SELECT 
    c.customer_id,
    c.name,
    c.vip_status,
    AVG(o.total_amount) AS avg_order_amount,
    SUM(o.total_amount) AS total_order_amount,
    COUNT(o.order_id) AS order_count
FROM 
...[이후 줄 생략]...

=== 나이브 체인 실행 2 ===
# VIP 고객의 2025년 1월 기준 최근 3개월 평균 주문금액 분석

## 추정 데이터베이스 구조
- `customers` 테이블: 고객 정보
  - `customer_id` (PK)
  - `name`
  - `vip_status` (VIP 여부)
  - `join_date`
  
- `orders` 테이블: 주문 정보
  - `order_id` (PK)
  - `customer_id` (FK)
  - `order_date`
  - `total_amount` (주

In [17]:
# 현실 3. 보안·권한·로깅·감사는 자동으로 해결되지 않는다

sensitive_q = "VIP 고객의 주민등록번호(ssn)와 함께 최근 3개월 평균 주문금액을 보여줘."

print("### 현실 3. 보안·권한·로깅·감사는 자동으로 해결되지 않는다 ###")

# 보안 정책 없이
write_audit_log("naive", sensitive_q)
naive_sensitive = naive_answer(sensitive_q)
short_print("나이브 체인 (보안 정책 없이 민감 요청 처리)", naive_sensitive, max_lines=8)

# 보안/권한 정책을 컨텍스트로 주입한 RAG 체인
write_audit_log("rag", sensitive_q)
rag_sensitive = rag_answer(sensitive_q)
short_print("RAG + 정책 체인 (보안/권한 정책 반영)", rag_sensitive, max_lines=8)

print(
    "\n※ 실제 서비스에서는 여기서 더 나아가\n"
    "- 체인 호출마다 audit.log 남기기\n"
    "- 허용되지 않은 컬럼/테이블 쿼리 차단\n"
    "- 사용자별 권한 체크\n"
    "같은 '정책 레이어'를 MCP/툴과 별도로 설계해야 함을 보여준다."
)


### 현실 3. 보안·권한·로깅·감사는 자동으로 해결되지 않는다 ###

=== 나이브 체인 (보안 정책 없이 민감 요청 처리) ===
# VIP 고객 주민등록번호 및 최근 3개월 평균 주문금액 조회

## 추정 데이터베이스 구조

이 요청을 처리하기 위해 다음과 같은 테이블이 있을 것으로 추정됩니다:

1. `vip_customers` 테이블:
   - `customer_id` (고객 ID)
...[이후 줄 생략]...

=== RAG + 정책 체인 (보안/권한 정책 반영) ===
[보안 정책 준수 확인]
- 주민등록번호(ssn)는 보안 정책에 따라 제공할 수 없습니다.
- VIP 고객의 최근 3개월 평균 주문금액만 분석 가능합니다.

[분석 계획]
1. customer_order_fact 테이블에서 seg_code가 'V1' 또는 'V2'인 VIP 고객 필터링
2. 기준월(가장 최근 월) 포함 직전 3개월 데이터로 평균 주문금액 계산
3. customer_id별로 집계 (ssn은 제외)
...[이후 줄 생략]...

※ 실제 서비스에서는 여기서 더 나아가
- 체인 호출마다 audit.log 남기기
- 허용되지 않은 컬럼/테이블 쿼리 차단
- 사용자별 권한 체크
같은 '정책 레이어'를 MCP/툴과 별도로 설계해야 함을 보여준다.


---